## Classifiers to Add:
Shot Value
Paint
Mid range
3 pointer
Catch and shoot


## Composite Features to Add:
Defender-distance ratio
Time pressure index
Early clock indicator
Creation load (log version)

In [ ]:
import pandas as pd

df_clean = pd.read_csv("data/processed/shot_logs_cleaned.csv")

# Shot clock as a fraction of the 24-second shot clock, with missing values set to 0
df_clean["shot_clock_pct"] = df_clean["shot_clock"] / 24
df_clean.loc[df_clean["shot_clock_missing"] == 1, "shot_clock_pct"] = 0


# Game clock as a fraction of the period's total time (720 seconds per quarter)
df_clean["game_clock_pct"] = df_clean["game_clock_sec"] / (df_clean["period"] * 720)


# Catch-and-shoot flag: no dribbles and minimal time with the ball
df_clean["catch_and_shoot"] = (
    (df_clean["dribbles"] == 0) & (df_clean["touch_time"] < 2)
).astype(int)


# Dribble pull-up flag: heavy ball-handling before the shot
df_clean["dribble_pull_up"] = (
    (df_clean["dribbles"] >= 3) & (df_clean["touch_time"] > 2)
).astype(int)


# Interaction term: defender proximity × shot distance
df_clean["def_dist_x_shot_dist"] = df_clean["close_def_dist"] * df_clean["shot_dist"]


# Quadratic shot distance to capture non-linear range penalty
df_clean["shot_dist_squared"] = df_clean["shot_dist"] ** 2


print("Features engineered. New columns:")
print([c for c in df_clean.columns if c not in [
    "player_name", "matchup", "location", "period", "game_clock_sec",
    "shot_clock", "shot_clock_missing", "dribbles", "touch_time",
    "shot_dist", "close_def_dist", "pts_type", "made"
]])

# save cleaned dataset with all engineered features
df_clean.to_csv("data/processed/shot_logs_cleaned.csv", index=False)
print("Saved to data/processed/shot_logs_cleaned.csv")